# 19 — Training Pipeline and Compute Pilot

Builds the dataloader, model, and training loop, then runs a short **timing pilot** on a small
subset to answer the one open question from earlier: does full 3D training actually fit Kaggle's
30 GPU-hr/week quota within the timeline, or is the 2.5D fallback needed.

**Model:** `r3d_18`, a 3D ResNet pretrained on Kinetics-400 (video). The depth axis of the CT volume
is treated as the time axis a video model expects — this is the standard way to get real pretrained
3D weights, since no 3D equivalent of ImageNet pretraining exists. The first conv layer is adapted
from 3-channel (RGB) to 1-channel (CT) input by averaging the pretrained RGB filter weights.

**Input:** the full preprocessed volume `(128, 160, 192)` — no patch extraction. At this size it's a
reasonable bet that it fits in GPU memory with a small batch size; the pilot below confirms this
rather than assuming it.

**Before running on Kaggle:** update `PROCESSED_DIR`, `SPLIT_PATH`, and `CHECKPOINT_DIR` below to
your Kaggle paths (typically `/kaggle/input/...` for read-only data and `/kaggle/working/...` for
anything written, since only `/kaggle/working` persists as downloadable output).

In [1]:
# ============================================================
# CELL 1 — IMPORTS AND CONFIGURATION
# ============================================================

import time
from pathlib import Path
import platform 

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models.video import r3d_18, R3D_18_Weights

# ------------------------------------------------------------
# Paths — UPDATE THESE for Kaggle before running there.
# ------------------------------------------------------------

PROCESSED_DIR = Path(r"D:\Pancreatic_Cancer_Thesis\data\processed")
SPLIT_PATH = PROCESSED_DIR / "split_assignment.csv"
CHECKPOINT_DIR = Path(r"D:\Pancreatic_Cancer_Thesis\checkpoints")

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Hyperparameters
# ------------------------------------------------------------

BATCH_SIZE = 2
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5

# Pilot settings on GPU — small subset, few epochs, just to time it.
PILOT_MAX_TRAIN_CASES = 24
PILOT_MAX_VAL_CASES = 8
PILOT_EPOCHS = 2

# Smoke-test settings on CPU — just enough to confirm the code runs
# end-to-end (shapes, loss, checkpointing). Not meant to produce
# meaningful timing numbers; r3d_18 on full 3D volumes is genuinely
# slow on CPU, so this keeps a local sanity check fast.
CPU_SMOKE_TEST_CASES = 3
CPU_SMOKE_TEST_EPOCHS = 1

# Real training settings (used once the pilot confirms feasibility).
FULL_TRAIN_EPOCHS = 40
KAGGLE_WEEKLY_GPU_HOURS = 30

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Real parallel workers only on an actual CUDA GPU running on Linux (Kaggle).
# Windows uses the "spawn" multiprocessing start method, which requires the
# Dataset class to be importable from a real module -- not defined in a
# notebook cell -- so workers crash immediately there regardless of whether
# a GPU is present. Force num_workers=0 on Windows to avoid that.
NUM_WORKERS = 2 if (DEVICE.type == "cuda" and platform.system() != "Windows") else 0

# On CPU, automatically shrink the pilot to a smoke test so local
# testing finishes in a reasonable time. On GPU, the full pilot
# settings above are used as-is (that's what produces the real
# Kaggle GPU-hour estimate in Cell 7).
if DEVICE.type != "cuda":
    PILOT_MAX_TRAIN_CASES = CPU_SMOKE_TEST_CASES
    PILOT_MAX_VAL_CASES = CPU_SMOKE_TEST_CASES
    PILOT_EPOCHS = CPU_SMOKE_TEST_EPOCHS

print("=" * 70)
print("TRAINING PIPELINE — CONFIGURATION")
print("=" * 70)
print("Device        :", DEVICE)
print("Processed dir :", PROCESSED_DIR)
print("Split file    :", SPLIT_PATH)
print("Checkpoints   :", CHECKPOINT_DIR)
print("Batch size    :", BATCH_SIZE)

if DEVICE.type == "cuda":
    print("\nGPU:", torch.cuda.get_device_name(0))
    print("Total GPU memory: "
          f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"\nPilot mode: full pilot ({PILOT_MAX_TRAIN_CASES} train / "
          f"{PILOT_MAX_VAL_CASES} val, {PILOT_EPOCHS} epochs) — timing "
          "numbers below are meaningful.")
else:
    print("\n\u26a0 No GPU detected.")
    print(f"Pilot mode: CPU SMOKE TEST ONLY ({PILOT_MAX_TRAIN_CASES} train / "
          f"{PILOT_MAX_VAL_CASES} val, {PILOT_EPOCHS} epoch). This just checks "
          "the code runs correctly -- re-run on a GPU (e.g. Kaggle) for real "
          "timing numbers before trusting Cell 7's estimate.")

TRAINING PIPELINE — CONFIGURATION
Device        : cpu
Processed dir : D:\Pancreatic_Cancer_Thesis\data\processed
Split file    : D:\Pancreatic_Cancer_Thesis\data\processed\split_assignment.csv
Checkpoints   : D:\Pancreatic_Cancer_Thesis\checkpoints
Batch size    : 2

⚠ No GPU detected.
Pilot mode: CPU SMOKE TEST ONLY (3 train / 3 val, 1 epoch). This just checks the code runs correctly -- re-run on a GPU (e.g. Kaggle) for real timing numbers before trusting Cell 7's estimate.


In [2]:
# ============================================================
# CELL 2 — LOAD SPLIT AND BUILD SUBSET LOADERS FOR THE PILOT
# ============================================================

if not SPLIT_PATH.exists():
    raise FileNotFoundError(f"Split file not found:\n{SPLIT_PATH}")

split_df = pd.read_csv(SPLIT_PATH)

train_df_full = split_df[split_df["split"] == "train"].reset_index(drop=True)
val_df_full = split_df[split_df["split"] == "validation"].reset_index(drop=True)

print("=" * 70)
print("SPLIT LOADED")
print("=" * 70)
print("Full train cases:", len(train_df_full))
print("Full val cases  :", len(val_df_full))

# Class balance in the training set, used for pos_weight below.
train_pdac = int((train_df_full["diagnosis"] == "PDAC").sum())
train_non_pdac = int((train_df_full["diagnosis"] == "non-PDAC").sum())
pos_weight_value = train_non_pdac / max(train_pdac, 1)

print(f"\nTrain class balance: {train_pdac} PDAC / {train_non_pdac} non-PDAC")
print(f"pos_weight for BCE loss: {pos_weight_value:.3f}")

SPLIT LOADED
Full train cases: 1563
Full val cases  : 338

Train class balance: 473 PDAC / 1090 non-PDAC
pos_weight for BCE loss: 2.304


In [3]:
# ============================================================
# CELL 3 — DATASET CLASS
# ============================================================


class PDACVolumeDataset(Dataset):
    """
    Loads a full preprocessed CT volume (128, 160, 192) and its
    PDAC / non-PDAC label. No patch extraction -- the volume is
    already a fixed-size pancreas-region crop.
    """

    def __init__(self, dataframe, processed_dir, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.processed_dir = Path(processed_dir)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = self.processed_dir / row["image_path"]
        volume = np.load(image_path).astype(np.float32)

        if self.augment and np.random.rand() < 0.5:
            # Left-right flip as a simple augmentation.
            volume = np.flip(volume, axis=2).copy()

        # Add a channel dimension: (D, H, W) -> (1, D, H, W)
        volume = np.expand_dims(volume, axis=0)

        label = 1.0 if row["diagnosis"] == "PDAC" else 0.0

        return (
            torch.from_numpy(volume),
            torch.tensor(label, dtype=torch.float32),
            row["study_id"],
        )


print("\u2713 PDACVolumeDataset defined.")

✓ PDACVolumeDataset defined.


In [4]:
# ============================================================
# CELL 4 — MODEL: KINETICS-PRETRAINED 3D RESNET, ADAPTED TO 1 CHANNEL
# ============================================================


def build_model(pretrained=True):
    weights = R3D_18_Weights.KINETICS400_V1 if pretrained else None
    model = r3d_18(weights=weights)

    # Adapt the stem's first conv from 3-channel (RGB video) to
    # 1-channel (CT). Average the pretrained RGB filters into a
    # single-channel filter so the pretrained weights aren't discarded.
    old_conv = model.stem[0]
    new_conv = nn.Conv3d(
        in_channels=1,
        out_channels=old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=False,
    )
    if pretrained:
        with torch.no_grad():
            new_conv.weight[:] = old_conv.weight.mean(dim=1, keepdim=True)
    model.stem[0] = new_conv

    # Replace the classification head for binary PDAC / non-PDAC.
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, 1)

    return model


model = build_model(pretrained=True).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print("=" * 70)
print("MODEL BUILT")
print("=" * 70)
print("Architecture   : r3d_18 (Kinetics-400 pretrained)")
print("Input channels : 1 (adapted from 3)")
print("Total params   :", f"{n_params:,}")

MODEL BUILT
Architecture   : r3d_18 (Kinetics-400 pretrained)
Input channels : 1 (adapted from 3)
Total params   : 33,147,969


In [5]:
# ============================================================
# CELL 5 — TRAINING / EVAL STEP AND CHECKPOINT HELPERS
# ============================================================


def run_epoch(model, loader, optimizer, criterion, device, scaler, train=True):
    model.train() if train else model.eval()

    total_loss = 0.0
    n_samples = 0
    start = time.time()

    for volumes, labels, _ in loader:
        volumes = volumes.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.set_grad_enabled(train):
            with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                logits = model(volumes).squeeze(1)
                loss = criterion(logits, labels)

            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * volumes.size(0)
        n_samples += volumes.size(0)

    elapsed = time.time() - start
    return total_loss / max(n_samples, 1), elapsed


def save_checkpoint(path, model, optimizer, epoch, best_val_loss):
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "best_val_loss": best_val_loss,
    }, path)


def load_checkpoint(path, model, optimizer, device):
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    return checkpoint["epoch"], checkpoint["best_val_loss"]


print("\u2713 Training step and checkpoint helpers defined.")

✓ Training step and checkpoint helpers defined.


In [6]:
# ============================================================
# CELL 6 — TIMING PILOT (RUN THIS BEFORE COMMITTING TO A FULL RUN)
# ============================================================

pilot_train_df = train_df_full.sample(
    n=min(PILOT_MAX_TRAIN_CASES, len(train_df_full)), random_state=42
).reset_index(drop=True)
pilot_val_df = val_df_full.sample(
    n=min(PILOT_MAX_VAL_CASES, len(val_df_full)), random_state=42
).reset_index(drop=True)

pilot_train_ds = PDACVolumeDataset(pilot_train_df, PROCESSED_DIR, augment=True)
pilot_val_ds = PDACVolumeDataset(pilot_val_df, PROCESSED_DIR, augment=False)

pilot_train_loader = DataLoader(
    pilot_train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
)
pilot_val_loader = DataLoader(
    pilot_val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
)

pilot_model = build_model(pretrained=True).to(DEVICE)
pilot_optimizer = torch.optim.AdamW(
    pilot_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
pilot_criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(pos_weight_value, device=DEVICE)
)
pilot_scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

run_label = "TIMING PILOT" if DEVICE.type == "cuda" else "CPU SMOKE TEST"

print("=" * 70)
print(f"{run_label} — {len(pilot_train_ds)} train / {len(pilot_val_ds)} val cases, "
      f"{PILOT_EPOCHS} epoch(s)")
print("=" * 70)

epoch_times = []

if DEVICE.type != "cuda":
    # A full multi-batch train+backward loop with r3d_18 at full resolution
    # (128,160,192) is too heavy for CPU-only RAM and can crash or thrash
    # the machine (as you just saw). A single forward-only pass is enough
    # to confirm shapes/dtypes/loss compute correctly. Real training and
    # timing happen on Kaggle GPU, never here.
    print("CPU detected -- running a single forward-pass check only "
          "(no backward/optimizer step). See comment above for why.")

    pilot_model.eval()
    volumes, labels, study_ids = next(iter(pilot_train_loader))

    print("\nBatch volumes shape:", volumes.shape)
    print("Batch labels       :", labels)
    print("Study IDs          :", study_ids)

    start = time.time()
    with torch.no_grad():
        logits = pilot_model(volumes.to(DEVICE)).squeeze(1)
        loss = pilot_criterion(logits, labels.to(DEVICE))
    elapsed = time.time() - start

    print(f"\nForward pass OK. Loss = {loss.item():.4f} ({elapsed:.1f}s)")
    print("\nCPU smoke test passed. Move to Kaggle GPU for the real "
          "training pilot -- do not attempt full CPU training here.")

else:
    torch.cuda.reset_peak_memory_stats()

    for epoch in range(1, PILOT_EPOCHS + 1):
        train_loss, train_time = run_epoch(
            pilot_model, pilot_train_loader, pilot_optimizer,
            pilot_criterion, DEVICE, pilot_scaler, train=True,
        )
        val_loss, val_time = run_epoch(
            pilot_model, pilot_val_loader, pilot_optimizer,
            pilot_criterion, DEVICE, pilot_scaler, train=False,
        )

        epoch_times.append(train_time)

        print(f"Epoch {epoch}/{PILOT_EPOCHS} "
              f"| train_loss={train_loss:.4f} ({train_time:.1f}s) "
              f"| val_loss={val_loss:.4f} ({val_time:.1f}s)")

    peak_mem_gb = torch.cuda.max_memory_allocated() / 1e9
    print(f"\nPeak GPU memory used: {peak_mem_gb:.2f} GB")

C:\Users\Dell\AppData\Local\Temp\ipykernel_8304\246082652.py:31: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  pilot_scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))


CPU SMOKE TEST — 3 train / 3 val cases, 1 epoch(s)
CPU detected -- running a single forward-pass check only (no backward/optimizer step). See comment above for why.

Batch volumes shape: torch.Size([2, 1, 128, 160, 192])
Batch labels       : tensor([0., 0.])
Study IDs          : ('100375_00001', '100738_00001')

Forward pass OK. Loss = 0.5775 (51.3s)

CPU smoke test passed. Move to Kaggle GPU for the real training pilot -- do not attempt full CPU training here.


In [7]:
# ============================================================
# CELL 7 — EXTRAPOLATE PILOT TIMING TO A REAL FULL TRAINING RUN
# ============================================================
if not epoch_times:
    print("No epoch timing data collected (CPU smoke-test mode ran a single "
          "forward pass only). Skipping extrapolation -- re-run on Kaggle GPU "
          "for the real feasibility estimate.")
else:
      avg_pilot_train_epoch_sec = np.mean(epoch_times)

      # Scale by how many batches a full-size epoch would have
      # relative to the pilot subset.
      scale_factor = len(train_df_full) / len(pilot_train_ds)
      est_full_epoch_sec = avg_pilot_train_epoch_sec * scale_factor

      est_full_epoch_min = est_full_epoch_sec / 60
      est_total_hours = (est_full_epoch_sec * FULL_TRAIN_EPOCHS) / 3600
      est_weeks_needed = est_total_hours / KAGGLE_WEEKLY_GPU_HOURS

      print("=" * 70)
      print("EXTRAPOLATED FULL TRAINING COST")
      print("=" * 70)
      print(f"Pilot avg epoch time (subset)   : {avg_pilot_train_epoch_sec:.1f} sec "
            f"over {len(pilot_train_ds)} cases")
      print(f"Scale factor to full train set  : {scale_factor:.1f}x "
            f"({len(train_df_full)} cases)")
      print(f"Estimated full epoch time       : {est_full_epoch_min:.1f} min")
      print(f"Estimated total for {FULL_TRAIN_EPOCHS} epochs   : {est_total_hours:.1f} GPU-hours")
      print(f"At {KAGGLE_WEEKLY_GPU_HOURS} GPU-hrs/week on Kaggle : "
            f"~{est_weeks_needed:.1f} weeks of quota")

      print("\n" + "-" * 70)
      if DEVICE.type != "cuda":
            print("\u26a0 This ran on CPU. Re-run on a GPU runtime before trusting these numbers.")
      elif est_weeks_needed <= 2:
            print("\u2713 Full 3D training looks feasible within the timeline. Proceed with 3D.")
      elif est_weeks_needed <= 4:
            print("\u26a0 Full 3D training is borderline. Consider fewer epochs, a smaller "
            "model, or the 2.5D fallback.")
      else:
            print("\u2717 Full 3D training does not fit the compute budget as configured. "
            "Switch to the 2.5D fallback or reduce FULL_TRAIN_EPOCHS / batch size.")

No epoch timing data collected (CPU smoke-test mode ran a single forward pass only). Skipping extrapolation -- re-run on Kaggle GPU for the real feasibility estimate.


## Next, once the pilot confirms feasibility

1. Set `PILOT_MAX_TRAIN_CASES` / `PILOT_MAX_VAL_CASES` to the full split sizes (or skip Cells 6-7
   and write a full training loop reusing `run_epoch`, `save_checkpoint`, `load_checkpoint` above).
2. Add per-epoch checkpointing (`save_checkpoint` every epoch) and a startup check that resumes
   from the latest checkpoint if one exists, so a run survives the 12-hour Kaggle session limit.
3. Use Kaggle's **Save & Run All (commit)** to run training as a background job that keeps going
   even if the local connection drops — relevant given the local power-cut situation.
4. Track validation loss per epoch and keep the best checkpoint separately, since that's the model
   used later for calibration and Grad-CAM evaluation, not necessarily the final epoch's weights.